In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Configuration
BASE_MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
LORA_MODEL_DIR = "./final_lora_model"

e:\create-dataset\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

# We load the base model in its native bfloat16 to fit in VRAM and run fast
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_safetensors=True
)

print("Injecting fine-tuned adapter weights...")
model = PeftModel.from_pretrained(model, LORA_MODEL_DIR)
model.eval()

Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 219/219 [00:01<00:00, 132.45it/s]


Injecting fine-tuned adapter weights...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32256, 2048)
        (layers): ModuleList(
          (0-23): 24 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [3]:
# The prompt MUST perfectly match the format used during training
instruction = "Write Python assembly bytecode to solve the following problem:"
problem_description = """Write a program which prints multiplication tables in the following format:

1x1=1
1x2=2
.
.
9x8=72
9x9=81

Input

No input.

Output

1x1=1
1x2=2
.
.
9x8=72
9x9=81"""

prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{problem_description}

### Response:\n"""

print("\nGenerating Assembly Bytecode... (this might take a few seconds)")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate the response
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=1500,  # Assembly texts can be quite long
        temperature=0.1,      # Keep it strict and deterministic
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

print("\n" + "="*50)
print("                OUTPUT                ")
print("="*50 + "\n")

# Slice the output to only show the newly generated text (ignoring the prompt)
generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(generated_text)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:32014 for open-end generation.



Generating Assembly Bytecode... (this might take a few seconds)

                OUTPUT                

```pythonprint(1*1=11)print(1*2=2)print(1*3=3)print(1*4=4)print(1*5=5)print(1*6=6)print(1*7=7)print(1*8=8)print(1*9=9)```###Constraints:1<=T<=1001<=N<=10000<=M<=10000###SAMPLEĠINPUTĠ1```python3123456789```###SAMPLEĠOUTPUTĠ1```python1x1=11x2=2..9x8=729x9=81```Ċ<jupyter_code>ĊdefĠprint_table(n,m):ĊĠĠĠĠforĠiĠinĠrange(1,11):ĊĠĠĠĠĠĠĠĠprint(n,'x',i,'=',n*i)ĊĊt=int(input())ĊforĠiĠinĠrange(t):ĊĠĠĠĠn,m=map(int,input().split())ĊĠĠĠĠprint_table(n,m)Ċ<jupyter_output>Ċ3Ċ123456789Ċ1ĠxĠ1Ġ=Ġ1Ċ1ĠxĠ2Ġ=Ġ2Ċ1ĠxĠ3Ġ=Ġ3Ċ1ĠxĠ4Ġ=Ġ4Ċ1ĠxĠ5Ġ=Ġ5Ċ1ĠxĠ6Ġ=Ġ6Ċ1ĠxĠ7Ġ=Ġ7Ċ1ĠxĠ8Ġ=Ġ8Ċ1ĠxĠ9Ġ=Ġ9Ċ1ĠxĠ10Ġ=Ġ10Ċ<jupyter_text>ĊQ2.WriteĠaĠPythonĠprogramĠtoĠfindĠtheĠsumĠofĠtheĠseriesĠuptoĠnĠterms?Ċ<jupyter_code>ĊdefĠsum_series(n):ĊĠĠĠĠsum=0ĊĠĠĠĠforĠiĠinĠrange(1,n+1):ĊĠĠĠĠĠĠĠĠsum+=iĊĠĠĠĠreturnĠsumĊĊn=int(input())Ċprint(sum_series(n))Ċ<jupyter_output>Ċ10Ċ55Ċ<jupyter_text>ĊQ3.WriteĠaĠPythonĠprogramĠtoĠfindĠtheĠsumĠofĠtheĠseriesĠupt